# Segmentación de edificaciones PeruSAT-1: RGB+NIR

Se comparan SegFormer MiT-B2, U-Net ResNet-50 y DeepLabV3+ ResNet-50 utilizando encoders preentrenados en ImageNet y un protocolo común de entrenamiento y evaluación. Los modelos utilizan imágenes RGB-NIR, pérdidas combinadas, rebalanceo de muestras y métricas de segmentación a nivel de píxel. Las predicciones se binarizan utilizando un umbral fijo de 0.5.

## 1. Entorno, Drive y datos

Se definen las rutas de trabajo y acceso al dataset.

### 1.1 Instalación de dependencias


In [ ]:
%pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124

%pip install \
    segmentation-models-pytorch==0.5.0 \
    pytorch-lightning==2.5.1 \
    torchmetrics==1.7.1 \
    timm==1.0.15 \
    transformers==4.51.3 \
    huggingface-hub==0.31.2 \
    albumentations==2.0.8 \
    albucore==0.0.24 \
    opencv-python-headless==4.11.0.86 \
    numpy==2.2.6 \
    scipy==1.15.3 \
    rasterio==1.4.3 \
    pandas==2.2.3 \
    matplotlib==3.10.1 \
    safetensors==0.5.3 \
    librosa==0.11.0

### 1.2 Montaje opcional de Drive y rutas


In [ ]:
# El montaje permite acceder a archivos privados almacenados en Google Drive.
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Opcional en Colab: ejecutar solo para extraer un archivo RAR.
!apt-get -qq update
!apt-get -qq install -y unrar

!mkdir -p /content/Dataset
!unrar x -o+ "RUTA_ARCHIVO_RAR" /content/Dataset/

In [ ]:
from pathlib import Path
import os
import json

# Dataset descomprimido localmente para que la lectura durante entrenamiento sea rápida.
DATA_ROOT = Path(
    os.environ.get(
        "PERUSAT1_DATA_ROOT",
        "/content/Dataset"
    )
)

# Artefactos persistentes.
PERSISTENT_ROOT = Path(
    os.environ.get("PERUSAT1_RUN_DIR", "runs")
)

PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

NIR_STATS_PATH = PERSISTENT_ROOT / "nir_stats.json"

# Se reanuda automáticamente desde last.ckpt; las rutas están separadas por arquitectura.


## 2. Preparacion de datos

Imports, constantes de mascara, clase dataset, aumentaciones y loaders.

### 2.1 Imports y constantes de mascara

Librerias principales y valores usados para interpretar las mascaras.

In [ ]:
import os
import gc
import re
import cv2
import numpy as np
import pandas as pd
import albumentations as A
import torch
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, ConcatDataset, get_worker_info
from torch.utils.data import Dataset as BaseDataset
from collections import defaultdict
import matplotlib.pyplot as plt
import rasterio
from IPython.display import display

NOINFO_VALUE = 0
BACKGROUND = 1
BUILDING = 3
EXCLUDE = 10
SEED = 42
THRESHOLD = 0.5

# Los índices raster son de base uno y corresponden al orden R, G, B, NIR.
BAND_INDICES = [1, 2, 3, 4]
IMAGE_SCALE = 255.0
RASTER_EXTENSIONS = {".tif", ".tiff"}
pl.seed_everything(SEED, workers=True)


### 2.2 Dataset personalizado RGB+NIR

Se ordenan los rásteres y se comprueba la correspondencia exacta entre imágenes y máscaras. Se realiza escalado `/255.0`, aplicado una sola vez al leer cada muestra. La normalización se realiza dentro del modelo.


In [ ]:
class Dataset(BaseDataset):
    """Se entregan imagen escalada (4,H,W), edificios (1,H,W) y validez (1,H,W)."""

    def __init__(self, images_dir, masks_dir, spatial_augmentation=None, rgb_augmentation=None):
        images_dir, masks_dir = Path(images_dir), Path(masks_dir)
        self.ids = sorted(
            path.name for path in images_dir.iterdir()
            if path.is_file() and path.suffix.lower() in RASTER_EXTENSIONS
        )
        if not self.ids:
            raise ValueError(f"No se encontraron imágenes raster en {images_dir}")
        if any(not name.startswith("img_") for name in self.ids):
            raise ValueError("Los nombres de las imágenes deben comenzar con img_.")
        mask_ids = ["mask_" + name[len("img_"):] for name in self.ids]
        available_masks = {
            path.name for path in masks_dir.iterdir()
            if path.is_file() and path.suffix.lower() in RASTER_EXTENSIONS
        }
        missing = set(mask_ids) - available_masks
        orphan = available_masks - set(mask_ids)
        if missing or orphan:
            raise ValueError(
                f"Correspondencia imagen/máscara incompleta: "
                f"faltantes={sorted(missing)[:10]}, sin imagen={sorted(orphan)[:10]}"
            )
        self.images_fps = [str(images_dir / name) for name in self.ids]
        self.masks_fps = [str(masks_dir / name) for name in mask_ids]
        self.spatial_augmentation = spatial_augmentation
        self.rgb_augmentation = rgb_augmentation
        self._worker_seed = None

    def __len__(self):
        return len(self.ids)

    def read_sample(self, i):
        """Lee RGB+NIR y la validez sin normalizar radiométricamente."""
        with rasterio.open(self.images_fps[i]) as src:
            if src.count < max(BAND_INDICES):
                raise ValueError(f"Falta una banda RGB+NIR en {self.images_fps[i]}")

            image = src.read(BAND_INDICES)

        image = np.transpose(image, (1, 2, 0))

        with rasterio.open(self.masks_fps[i]) as src:
            mask = src.read(1)

        if image.shape[:2] != mask.shape:
            raise ValueError(f"Dimensiones incompatibles en {self.ids[i]}")

        valid = (mask != NOINFO_VALUE) & (mask != EXCLUDE)
        y = ((mask == BUILDING) & valid).astype(np.uint8)

        image[~valid] = 0

        return image, y, valid.astype(np.uint8)

    def __getitem__(self, i):
        # Albumentations mantiene generadores propios; cada worker recibe una semilla distinta.
        worker = get_worker_info()
        if worker is not None and self._worker_seed != worker.seed:
            self._worker_seed = worker.seed
            for offset, augmentation in enumerate((self.spatial_augmentation, self.rgb_augmentation)):
                if augmentation is not None:
                    augmentation.set_random_seed((worker.seed + offset) % (2**32))
        image, y, valid = self.read_sample(i)
        if self.spatial_augmentation is not None:
            sample = self.spatial_augmentation(image=image, mask=y, valid=valid)
            image, y, valid = sample["image"], sample["mask"], sample["valid"]
        if self.rgb_augmentation is not None:
            image = image.copy()
            image[..., :3] = self.rgb_augmentation(image=image[..., :3].copy())["image"]

        image = np.transpose(image, (2, 0, 1)).astype(np.float32) / IMAGE_SCALE
        image = np.ascontiguousarray(image)

        y = np.ascontiguousarray(y[None], dtype=np.float32)
        valid = np.ascontiguousarray(valid[None], dtype=np.float32)

        return image, y, valid


### 2.3 Aumentación de datos

Las transformaciones geométricas comparten los mismos parámetros para las cuatro bandas, la máscara y la validez. Brillo, contraste y desenfoque afectan únicamente a RGB.


In [ ]:
def get_training_spatial_augmentation():
    train_transform = [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            scale_limit=0.10, rotate_limit=0, shift_limit=0.05, p=0.7,
            border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0,
            mask_interpolation=cv2.INTER_NEAREST,
        ),
        A.PadIfNeeded(
            min_height=512, min_width=512,
            border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0,
        ),
    ]
    return A.Compose(train_transform, additional_targets={"valid": "mask"}, seed=SEED)


def get_training_rgb_augmentation():
    return A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.25),
        A.GaussianBlur(blur_limit=(3, 3), p=0.10),
    ], seed=SEED + 1)


def get_validation_augmentation():
    return A.Compose([
        A.PadIfNeeded(
            min_height=512, min_width=512,
            border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=0,
        ),
    ], additional_targets={"valid": "mask"}, seed=SEED)


### 2.4 Estadísticas NIR y construcción de datasets

La media y desviación poblacional de NIR se calculan sobre todos los píxeles válidos de train, tras el escalado, sin aumentación ni repeticiones. Se acumulan momentos en float64 sin cargar el conjunto completo en memoria. Los mismos valores se guardan en los checkpoints y se utilizan en los tres modelos, validación y test. Los splits permanecen en `train`, `val` y `test`.


In [ ]:
def compute_nir_stats(dataset):
    """Se combinan momentos de NIR ponderados por el número de píxeles válidos."""
    count, mean, m2 = 0, 0.0, 0.0
    for i in range(len(dataset)):
        image, _, valid = dataset.read_sample(i)
        values = (
            image[..., 3][valid.astype(bool)]
            .astype(np.float64)
            / IMAGE_SCALE
        )
        n = values.size
        if n == 0:
            continue
        patch_mean = float(values.mean())
        patch_m2 = float(np.square(values - patch_mean).sum())
        delta = patch_mean - mean
        total = count + n
        mean += delta * n / total
        m2 += patch_m2 + delta * delta * count * n / total
        count = total
    if count == 0:
        raise ValueError("Train no contiene píxeles válidos para calcular NIR.")
    std = float(np.sqrt(m2 / count))
    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        raise ValueError("Las estadísticas NIR no son finitas o la desviación es cero.")
    return {"mean": mean, "std": std, "valid_pixels": int(count)}

if NIR_STATS_PATH.exists():

    with open(NIR_STATS_PATH, "r") as f:
        NIR_STATS = json.load(f)

    print("Estadísticas NIR cargadas:", NIR_STATS)

else:

    print("No existen estadísticas NIR. Calculando...")

    train_for_stats = Dataset(
        DATA_ROOT / "train" / "images_norm",
        DATA_ROOT / "train" / "masks",
    )

    NIR_STATS = compute_nir_stats(train_for_stats)

    del train_for_stats

    with open(NIR_STATS_PATH, "w") as f:
        json.dump(NIR_STATS, f, indent=4)

    print("Estadísticas NIR calculadas y guardadas:", NIR_STATS)
    print("Archivo:", NIR_STATS_PATH)


def build_datasets():
    train_dataset = Dataset(
        DATA_ROOT / "train" / "images_norm", DATA_ROOT / "train" / "masks",
        spatial_augmentation=get_training_spatial_augmentation(),
        rgb_augmentation=get_training_rgb_augmentation(),
    )
    valid_dataset = Dataset(
        DATA_ROOT / "val" / "images_norm", DATA_ROOT / "val" / "masks",
        spatial_augmentation=get_validation_augmentation(),
    )
    test_dataset = Dataset(
        DATA_ROOT / "test" / "images_norm", DATA_ROOT / "test" / "masks",
        spatial_augmentation=get_validation_augmentation(),
    )
    return train_dataset, valid_dataset, test_dataset


### 2.5 Rebalanceo por hard mining

Repeticion controlada de origenes especificos para reforzar ejemplos dificiles durante el entrenamiento.

In [ ]:
repeat_map = {
    "Recortada_IMG_PER1_20220913145908_ORT_PMS_000659": 1,
    "IMG_PER1_20230531145858_ORT_PMS_000054_repasado_sara": 1,
    "IMG_PER1_20221119153338_ORT_PMS_000055_rural": 2,
    "IMG_PER1_20240816153927_ORT_PMS_000137_repasado_rural": 1,
    "IMG_PER1_20250302151959_ORT_PMS_000376_rural": 1,
    "IMG_PER1_20250302151959_ORT_PMS_000376_urbano":1,
    "IMG_PER1_20161031153102_ORT_PMS_001517_repasado_lambayeque_mar_urbano":1,
}


def balance_train_dataset(train_dataset):
    """Define el número de copias extra por origen."""
    origin_to_indices = defaultdict(list)
    for i, tile_id in enumerate(train_dataset.ids):
        for origin_name in repeat_map:
            if origin_name.lower() in tile_id.lower():
                origin_to_indices[origin_name].append(i)
    extra_subsets = []
    for origin_name, n_extra_copies in repeat_map.items():
        idxs = origin_to_indices[origin_name]
        print(f"{origin_name}: {len(idxs)} parches, {n_extra_copies} copias extra")
        if not idxs:
            print(f"[AVISO] No se encontraron parches de train para: {origin_name}")
            continue
        hard_subset = Subset(train_dataset, idxs)
        extra_subsets.extend([hard_subset] * n_extra_copies)
    train_dataset_balanced = (
        ConcatDataset([train_dataset] + extra_subsets) if extra_subsets else train_dataset
    )
    print("Tamaño train antes del rebalanceo:", len(train_dataset))
    print("Tamaño train final:", len(train_dataset_balanced))
    return train_dataset_balanced


### 2.6 DataLoaders

Configuracion de lotes, paralelismo y lectura eficiente para cada particion.

In [ ]:
# Configuración de lotes y workers para los DataLoaders
TRAIN_BATCH_SIZE = 96
VALID_BATCH_SIZE = 64
TEST_BATCH_SIZE = 16
NUM_WORKERS = 8


def build_dataloaders(train_dataset, valid_dataset, test_dataset):
    train_dataset_balanced = balance_train_dataset(train_dataset)
    worker_options = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        worker_options.update(persistent_workers=True, prefetch_factor=2)
    train_loader = DataLoader(
        train_dataset_balanced, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(SEED), **worker_options,
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=VALID_BATCH_SIZE, shuffle=False,
        generator=torch.Generator().manual_seed(SEED + 1), **worker_options,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False,
        generator=torch.Generator().manual_seed(SEED + 2), **worker_options,
    )
    return train_loader, valid_loader, test_loader

## 3. Modelos y entrenamiento

Se utiliza un único módulo Lightning y las mismas funciones de entrenamiento, validación y test.

### 3.1 Épocas y learning rates


In [ ]:
MAX_EPOCHS = 150
LEARNING_RATES = {
    "segformer_mitb2": 5e-5,
    "unet_resnet50": 5e-5,
    "deeplabv3plus_resnet50": 5e-5,
}

### 3.2 LightningModule común

Se define un LightningModule común para las tres arquitecturas, utilizando imágenes de cuatro bandas (RGB-NIR) y segmentación binaria de construcciones. Los encoders utilizan pesos preentrenados en ImageNet, adaptando automáticamente la primera capa para recibir cuatro canales. Las bandas RGB se normalizan con las estadísticas correspondientes a ImageNet, mientras que la banda NIR utiliza la media y desviación estándar calculadas sobre el conjunto de entrenamiento.

La función de pérdida combina Focal Loss, Tversky Loss y una pérdida auxiliar de clasificación a nivel de parche. Adicionalmente, se aplica una mayor ponderación a los píxeles cercanos a los bordes de las construcciones. Los píxeles marcados como inválidos se excluyen del cálculo de las métricas.
Durante el entrenamiento se registran IoU, F1-score, F0.5-score, precisión y recall mediante TorchMetrics. La optimización utiliza AdamW y un scheduler ReduceLROnPlateau basado en el F0.5-score de validación.


In [ ]:
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from torchmetrics.classification import BinaryJaccardIndex, BinaryF1Score, BinaryFBetaScore, BinaryPrecision, BinaryRecall


def _gaussian_kernel2d(kernel_size: int, sigma: float, device, dtype):
    ax = torch.arange(kernel_size, device=device, dtype=dtype) - (kernel_size - 1) / 2.0
    xx, yy = torch.meshgrid(ax, ax, indexing="ij")
    k = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    k = k / (k.sum() + 1e-12)
    return k.view(1, 1, kernel_size, kernel_size)


def boundary_weight_map(y01: torch.Tensor, valid: torch.Tensor, sigma=3.0, c=200.0, kernel_size=13):
    """
    y01:   (B,1,H,W) float {0,1}
    valid: (B,1,H,W) float {0,1}
    retorna w: (B,1,H,W) float, con w=0 en pixeles ignorados
    """
    # Los bordes de la pérdida se calculan sobre la zona válida.
    y = y01 * valid

    # El gradiente local delimita los bordes únicamente para ponderar la pérdida.
    dil = F.max_pool2d(y, kernel_size=3, stride=1, padding=1)
    ero = 1.0 - F.max_pool2d(1.0 - y, kernel_size=3, stride=1, padding=1)
    edge = (dil - ero).clamp(0, 1)

    k = _gaussian_kernel2d(kernel_size, sigma, device=y.device, dtype=y.dtype)
    blurred = F.conv2d(edge, k, padding=kernel_size // 2)

    w = (1.0 + c * blurred) * valid
    return w


def weighted_tversky_loss(logits, y, weight_map, alpha=0.7, beta=0.3, eps=1e-6):
    """
    alpha: penaliza FP
    beta:  penaliza FN
    weight_map: mapa de pesos pixel a pixel
    """
    p = torch.sigmoid(logits)
    y = y.float()
    weight_map = weight_map.float()

    tp = (weight_map * p * y).sum(dim=(1, 2, 3))
    fp = (weight_map * p * (1 - y)).sum(dim=(1, 2, 3))
    fn = (weight_map * (1 - p) * y).sum(dim=(1, 2, 3))

    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)

    return 1.0 - tversky.mean()


def focal_loss(logits, y, weight_map, gamma=2.0, alpha=0.75, eps=1e-6):
    p = torch.sigmoid(logits)
    bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
    p_t = p * y + (1 - p) * (1 - y)
    alpha_t = alpha * y + (1 - alpha) * (1 - y)
    focal = alpha_t * (1 - p_t) ** gamma * bce
    return (focal * weight_map).sum() / (weight_map.sum() + eps)

class BuildingModel(pl.LightningModule):
    def __init__(
        self,
        arch,
        encoder_name,
        in_channels,
        nir_mean,
        nir_std,
        out_classes=1,
        lr=5e-5,
        encoder_weights="imagenet",
        use_boundary_weighting=True,
        bw_sigma=3.0,
        bw_c=40,
        bw_kernel_size=13,
        **kwargs
    ):
        super().__init__()
        if in_channels != 4 or out_classes != 1:
            raise ValueError("Se requiere segmentación binaria con cuatro bandas.")
        if not np.isfinite(nir_mean) or not np.isfinite(nir_std) or nir_std <= 0:
            raise ValueError("Se requieren estadísticas NIR válidas calculadas en train.")
        self.save_hyperparameters()

        self.model = smp.create_model(
            arch,
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=in_channels,
            classes=out_classes,
            **kwargs,
        )

        params = smp.encoders.get_preprocessing_params(encoder_name, pretrained="imagenet")
        if params["input_space"] != "RGB" or list(params["input_range"]) != [0, 1]:
            raise ValueError("La normalización RGB del encoder no coincide con el escalado.")
        self.register_buffer(
            "input_mean", torch.tensor([*params["mean"], nir_mean], dtype=torch.float32).view(1, 4, 1, 1)
        )
        self.register_buffer(
            "input_std", torch.tensor([*params["std"], nir_std], dtype=torch.float32).view(1, 4, 1, 1)
        )
        if not all(parameter.requires_grad for parameter in self.model.encoder.parameters()):
            raise RuntimeError("El encoder debe permanecer entrenable.")

        encoder_out_channels = self.model.encoder.out_channels[-1]
        self.tile_classifier = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d(1),
            torch.nn.Flatten(),
            torch.nn.Linear(encoder_out_channels, 1)
        )
        self.aux_bce = torch.nn.BCEWithLogitsLoss()

        for stage in ["train", "valid", "test"]:
            setattr(self, f"{stage}_iou", BinaryJaccardIndex())
            setattr(self, f"{stage}_f1", BinaryF1Score())
            setattr(self, f"{stage}_fbeta", BinaryFBetaScore(beta=0.5))
            setattr(self, f"{stage}_precision", BinaryPrecision())
            setattr(self, f"{stage}_recall", BinaryRecall())

    def _log_segmentation_metrics(self, stage, pred, y, valid):
        valid_mask = valid.bool().view(-1)

        if valid_mask.sum() == 0:
            return

        pred_m = pred.bool().view(-1)[valid_mask]
        y_m = y.bool().view(-1)[valid_mask]

        metric_names = ["iou", "f1", "fbeta", "precision", "recall"]
        for name in metric_names:
            metric = getattr(self, f"{stage}_{name}")
            metric.update(pred_m, y_m)
            self.log(
                f"{stage}_{name}",
                metric,
                prog_bar=name in ["iou", "f1", "fbeta"],
                on_step=False,
                on_epoch=True,
            )

    def forward(self, image, valid=None):
        image = (image - self.input_mean) / self.input_std

        if valid is not None:
            image = image * valid

        features = self.model.encoder(image)
        decoder_output = self.model.decoder(features)
        seg_logits = self.model.segmentation_head(decoder_output)
        tile_logit = self.tile_classifier(features[-1])
        return seg_logits, tile_logit

    def shared_step(self, batch, stage):
        image, y, valid = batch
        image = image.float()
        y = y.float()
        valid = valid.float()

        logits, tile_logit = self.forward(image, valid)

        has_building = (y * valid).amax(dim=(1, 2, 3)).unsqueeze(1)

        if self.hparams.use_boundary_weighting:
            weight_map = boundary_weight_map(
                y, valid,
                sigma=self.hparams.bw_sigma,
                c=self.hparams.bw_c,
                kernel_size=self.hparams.bw_kernel_size
            )
        else:
            weight_map = valid

        floss = focal_loss(
            logits, y, weight_map,
            gamma=2.0,
            alpha=0.25
        )

        tloss = weighted_tversky_loss(
            logits, y, weight_map,
            alpha=0.7,
            beta=0.3
        )

        aux_loss = self.aux_bce(tile_logit, has_building)

        loss = 2.0 * floss + 1.0 * tloss + 0.2 * aux_loss

        pred = torch.sigmoid(logits) > THRESHOLD

        self.log(f"{stage}_loss",      loss,      prog_bar=True,  on_step=False, on_epoch=True)
        self.log(f"{stage}_floss",     floss,     prog_bar=False, on_step=False, on_epoch=True)
        self.log(f"{stage}_tloss",     tloss,     prog_bar=False, on_step=False, on_epoch=True)
        self.log(f"{stage}_aux_loss",  aux_loss,  prog_bar=False, on_step=False, on_epoch=True)
        self.log(f"{stage}_wmean",     weight_map.mean(), prog_bar=False, on_step=False, on_epoch=True)
        self.log(f"{stage}_wmax",      weight_map.max(),  prog_bar=False, on_step=False, on_epoch=True)

        self._log_segmentation_metrics(stage, pred, y, valid)
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        self.shared_step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",        # Se maximiza F0.5 de validación.
            factor=0.5,
            patience=4,
            threshold=0.002,
            threshold_mode="rel",
            cooldown=1,
            min_lr=1e-6
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "valid_fbeta",
                "interval": "epoch",
                "frequency": 1,
            },
        }


### 3.3 Callbacks, logger y checkpoints por arquitectura

Se configura early stopping, el registro de métricas y el almacenamiento del mejor checkpoint de cada arquitectura según el F0.5-score de validación. El entrenamiento se reanuda automáticamente si existe checkpoints/last.ckpt para esa arquitectura. Usar un PERSISTENT_ROOT nuevo para iniciar un experimento independiente.

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger

In [ ]:
MODEL_CONFIGS = {
    "segformer_mitb2": {"arch": "Segformer", "encoder_name": "mit_b2", "label": "SegFormer MiT-B2"},
    "unet_resnet50": {"arch": "Unet", "encoder_name": "resnet50", "label": "U-Net ResNet-50"},
    "deeplabv3plus_resnet50": {"arch": "DeepLabV3Plus", "encoder_name": "resnet50", "label": "DeepLabV3+ ResNet-50"},
}
best_model_paths = {}
runs = {}
validation_results = {}
test_results = {}


### 3.4  Entrenamiento común

Cada arquitectura reconstruye sus datasets, aumentaciones y dataloaders después de fijar la semilla.


In [ ]:
import torch
torch.set_float32_matmul_precision("high")

In [ ]:
def train_architecture(model_name):
    pl.seed_everything(SEED, workers=True)
    config = MODEL_CONFIGS[model_name]

    # Se invalidan las referencias anteriores al iniciar otra corrida de la misma arquitectura.
    for registry in (best_model_paths, runs, validation_results, test_results):
        registry.pop(model_name, None)

    train_dataset, valid_dataset, test_dataset = build_datasets()
    train_loader, valid_loader, test_loader = build_dataloaders(
        train_dataset, valid_dataset, test_dataset
    )

    # ==========================================================
    # DIRECTORIOS PERSISTENTES Y REANUDACIÓN AUTOMÁTICA
    # ==========================================================

    arch_dir = PERSISTENT_ROOT / model_name
    ckpt_dir = arch_dir / "checkpoints"
    log_dir = arch_dir / "logs"

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)

    last_checkpoint = ckpt_dir / "last.ckpt"

    if last_checkpoint.exists():
        resume_path = str(last_checkpoint)
        print(f"Reanudando {model_name} desde:")
        print(resume_path)
    else:
        resume_path = None
        print(f"Iniciando entrenamiento nuevo para {model_name}.")

    csv_logger = CSVLogger(
        save_dir=str(log_dir),
        name="sessions",
    )

    run_dir = arch_dir

    # ==========================================================

    checkpoint_callback = ModelCheckpoint(
        dirpath=str(ckpt_dir),
        monitor="valid_fbeta",
        mode="max",
        save_top_k=1,
        save_last=True,
        save_weights_only=False,
        filename=(
            "best-"
            + model_name
            + "-4ch-{epoch:02d}-{valid_fbeta:.4f}"
        ),
        auto_insert_metric_name=False,
    )

    early_stop_callback = EarlyStopping(
        monitor="valid_fbeta",
        mode="max",
        patience=30,
        min_delta=0.0005,
        verbose=True,
    )

    model = BuildingModel(
        config["arch"],
        config["encoder_name"],
        in_channels=4,
        out_classes=1,
        nir_mean=NIR_STATS["mean"],
        nir_std=NIR_STATS["std"],
        lr=LEARNING_RATES[model_name],
        encoder_weights="imagenet",
        use_boundary_weighting=True,
        bw_sigma=3.0,
        bw_c=40,
        bw_kernel_size=13,
    )

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        callbacks=[checkpoint_callback, early_stop_callback],
        accelerator="gpu",
        devices=1,
        precision="16-mixed",
        log_every_n_steps=1,
        logger=csv_logger,
    )

    # ==========================================================
    # ENTRENAMIENTO / REANUDACIÓN
    # ==========================================================

    trainer.fit(
        model,
        train_dataloaders=train_loader,
        val_dataloaders=valid_loader,
        ckpt_path=resume_path,
    )

    # ==========================================================
    # EXPORTAR EL MEJOR MODELO A .pth
    # VA EXACTAMENTE AQUÍ
    # ==========================================================

    best_path = checkpoint_callback.best_model_path

    best_model = BuildingModel.load_from_checkpoint(
        best_path,
        map_location="cpu",
        encoder_weights=None,
        strict=True,
    )

    export_path = arch_dir / f"best-{model_name}-4ch.pth"

    torch.save(
        {
            "state_dict": best_model.state_dict(),
            "hyper_parameters": dict(best_model.hparams),
            "band_indices": BAND_INDICES,
            "image_scale": IMAGE_SCALE,
            "nir_stats": NIR_STATS,
        },
        export_path,
    )

    print("Modelo exportado:", export_path)

    del best_model

    best_model_paths[model_name] = best_path

    runs[model_name] = {
        "run_dir": arch_dir,
        "checkpoint_dir": ckpt_dir,
        "best_checkpoint": best_path,
        "last_checkpoint": str(last_checkpoint),
        "export_path": export_path,
    }

    return runs[model_name]

### 3.5 SegFormer MiT-B2

Se entrena SegFormer con encoder MiT-B2, ImageNet y las cuatro bandas.


In [ ]:
segformer_run = train_architecture("segformer_mitb2")


### 3.6 U-Net con ResNet-50

Se utiliza el protocolo común y se emplea el encoder ResNet-50 con ImageNet y RGB+NIR.


In [ ]:
unet_run = train_architecture("unet_resnet50")


### 3.7 DeepLabV3+ con ResNet-50

Se utiliza el protocolo común y se emplea el encoder ResNet-50 con ImageNet y RGB+NIR.

In [ ]:
deeplab_run = train_architecture("deeplabv3plus_resnet50")


## 4. Mejor checkpoint y validación

Se carga el mejor checkpoint de cada arquitectura, seleccionado exclusivamente por F0.5 de validación. Las estadísticas NIR están incluidas en los parámetros y buffers del checkpoint. No se realiza test durante esta selección.


In [ ]:
def get_best_checkpoint(model_name):
    ckpt_dir = (
        PERSISTENT_ROOT
        / model_name
        / "checkpoints"
    )

    candidates = list(
        ckpt_dir.glob(
            f"best-{model_name}-4ch-*.ckpt"
        )
    )

    if not candidates:
        raise FileNotFoundError(
            f"No existe un checkpoint best "
            f"para {model_name} en {ckpt_dir}"
        )

    def extract_fbeta(path):
        match = re.search(
            r"-(\d+\.\d+)\.ckpt$",
            path.name,
        )

        if match is None:
            raise ValueError(
                f"No se pudo extraer valid_fbeta "
                f"de {path.name}"
            )

        return float(match.group(1))

    best_path = max(
        candidates,
        key=extract_fbeta,
    )

    return best_path

In [ ]:
def load_best_model(model_name):
    best_path = get_best_checkpoint(model_name)

    model = BuildingModel.load_from_checkpoint(
        str(best_path),
        map_location="cpu",
        encoder_weights=None,
        strict=True,
    )

    config = MODEL_CONFIGS[model_name]

    if (
        model.hparams.arch != config["arch"]
        or model.hparams.encoder_name != config["encoder_name"]
    ):
        raise ValueError(
            f"El checkpoint no corresponde a {model_name}."
        )

    return model.eval()


def evaluate_model(model, dataloader, stage):
    trainer = pl.Trainer(
        accelerator="gpu", devices=1, precision="16-mixed",
        logger=False, enable_checkpointing=False,
    )
    if stage == "valid":
        return trainer.validate(model, dataloaders=dataloader, verbose=False)[0]
    if stage == "test":
        return trainer.test(model, dataloaders=dataloader, verbose=False)[0]
    raise ValueError(f"Etapa de evaluación no admitida: {stage}")


### 4.1 Validación de los checkpoints seleccionados

Los resultados corresponden a los objetos TorchMetrics acumulados sobre los píxeles válidos de todo el conjunto.


In [ ]:
for model_name in MODEL_CONFIGS:
    pl.seed_everything(SEED, workers=True)
    valid_dataset = Dataset(
        DATA_ROOT / "val" / "images_norm", DATA_ROOT / "val" / "masks",
        spatial_augmentation=get_validation_augmentation(),
    )
    valid_loader = DataLoader(
        valid_dataset, batch_size=VALID_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(SEED + 1),
    )
    model2 = load_best_model(model_name)
    validation_results[model_name] = evaluate_model(model2, valid_loader, "valid")
    print(MODEL_CONFIGS[model_name]["label"], validation_results[model_name])
    model2.cpu()
    del model2, valid_loader
    gc.collect()
    torch.cuda.empty_cache()


## 5. Test explícito y tabla comparativa

Esta sección evalúa test una vez por arquitectura, después de la selección por validación y con el checkpoint registrado para ese modelo. Los resultados se obtienen de las métricas TorchMetrics finales; no se promedian métricas de batches ni se ajusta el umbral. La tabla reúne precisión, recall, F1, F0.5 e IoU de validación y test.


In [ ]:
for model_name in MODEL_CONFIGS:
    if model_name not in validation_results:
        raise RuntimeError(f"Falta la validación del checkpoint seleccionado para {model_name}.")
    pl.seed_everything(SEED, workers=True)
    test_dataset = Dataset(
        DATA_ROOT / "test" / "images_norm", DATA_ROOT / "test" / "masks",
        spatial_augmentation=get_validation_augmentation(),
    )
    test_loader = DataLoader(
        test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        generator=torch.Generator().manual_seed(SEED + 2),
    )
    model2 = load_best_model(model_name)
    test_results[model_name] = evaluate_model(model2, test_loader, "test")
    print(MODEL_CONFIGS[model_name]["label"], test_results[model_name])
    model2.cpu()
    del model2, test_loader
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
comparison_rows = []

for model_name, config in MODEL_CONFIGS.items():

    best_path = get_best_checkpoint(model_name)

    row = {
        "modelo": config["label"],
        "checkpoint": str(best_path),
    }

    for stage, results in (
        ("valid", validation_results),
        ("test", test_results),
    ):
        for metric in ("precision", "recall", "f1", "fbeta", "iou"):

            metric_key = f"{stage}_{metric}"

            if metric_key not in results[model_name]:
                raise RuntimeError(
                    f"Falta la métrica global {metric_key} de {model_name}."
                )

            value = float(results[model_name][metric_key])

            if not np.isfinite(value):
                raise ValueError(
                    f"La métrica {metric_key} de {model_name} no es finita."
                )

            row[metric_key] = value

    comparison_rows.append(row)

    model_dir = PERSISTENT_ROOT / model_name

    pd.DataFrame([row]).to_csv(
        model_dir / "pixel_metrics.csv",
        index=False,
    )

comparison_table = pd.DataFrame(comparison_rows)

display(
    comparison_table
    .drop(columns="checkpoint")
    .round(4)
)

comparison_path = (
    PERSISTENT_ROOT / "comparison_pixel_metrics.csv"
)

comparison_table.to_csv(
    comparison_path,
    index=False,
)

print("Tabla comparativa:", comparison_path)

## 6. Visualizacion cualitativa

Seleccion de parches y generacion de figuras comparativas para inspeccionar predicciones.

In [ ]:
SELECTED_QUALITATIVE_PATCHES = [
          {"department":"Arequipa",
      "patch_name":"img_IMG_PER1_20230531145858_ORT_PMS_000054_repasado_wilder_000680.tif"},
        {
        "department": "Cajamarca",
        "patch_name": "img_IMG_PER1_20200107153646_ORT_PMS_000626_repasado_rural_001572.tif", #ok
    },
    {
        "department": "Piura",
        "patch_name": "img_IMG_PER1_20240306154256_ORT_PMS_000191_repasado_piura_urbano_000072.tif",
    },
         {  "department": "Ucayali",
        "patch_name": "img_IMG_PER1_20220828152001_ORT_PMS_000672_repasado_urbano_000066.tif", #ok
    },
]

USE_AUTO_CROP = False
CROP_MARGIN_PX = 45

def _safe_filename(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")

def _as_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def _find_dataset_index_by_patch(dataset, patch_name):
    ids = dataset.ids
    basenames = [os.path.basename(str(x)) for x in ids]

    if patch_name in basenames:
        return basenames.index(patch_name)

    matches = [i for i, name in enumerate(basenames) if patch_name in name or name in patch_name]
    if len(matches) == 1:
        return matches[0]

    raise ValueError(f"No se encontró una coincidencia única para: {patch_name}")

def _rgb_for_display(image_np, valid_mask):
    rgb = np.transpose(image_np[:3], (1, 2, 0)).astype(float)
    rgb = np.clip(rgb, 0, 1)
    rgb = rgb.copy()
    rgb[~valid_mask] = 0.72
    return rgb

def _crop_slices_from_masks(masks, margin=45):
    union = np.zeros_like(masks[0], dtype=bool)
    for mask in masks:
        union |= mask.astype(bool)

    if not union.any():
        return slice(None), slice(None)

    ys, xs = np.where(union)
    y0 = max(int(ys.min()) - margin, 0)
    y1 = min(int(ys.max()) + margin + 1, union.shape[0])
    x0 = max(int(xs.min()) - margin, 0)
    x1 = min(int(xs.max()) + margin + 1, union.shape[1])
    return slice(y0, y1), slice(x0, x1)

def _overlay_mask(ax, rgb, mask, color_rgba):
    ax.imshow(rgb)
    layer = np.zeros((*mask.shape, 4), dtype=float)
    layer[mask.astype(bool)] = color_rgba
    ax.imshow(layer)
    if mask.any():
        ax.contour(mask.astype(float), levels=[0.5], colors=[color_rgba[:3]], linewidths=0.45)
    ax.axis("off")


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAMES = list(MODEL_CONFIGS)

if not MODEL_NAMES:
    raise ValueError("No hay modelos configurados.")

for model_name in MODEL_NAMES:
    if model_name not in test_results:
        raise RuntimeError(
            f"Falta la evaluación explícita de test para {model_name}."
        )

pl.seed_everything(SEED, workers=True)

dataset = Dataset(
    DATA_ROOT / "test" / "images_norm",
    DATA_ROOT / "test" / "masks",
    spatial_augmentation=get_validation_augmentation(),
)

rows = []

# Los parches y sus recortes se preparan una sola vez para todos los modelos.
for item in SELECTED_QUALITATIVE_PATCHES:
    idx = _find_dataset_index_by_patch(dataset, item["patch_name"])
    image, y, valid = dataset[idx]

    image_np = _as_numpy(image)
    y_np = _as_numpy(y)
    valid_np = _as_numpy(valid)

    valid_mask = valid_np[0] > 0.5
    gt_mask = (y_np[0] > 0.5) & valid_mask
    rgb = _rgb_for_display(image_np, valid_mask)

    if USE_AUTO_CROP:
        ys, xs = _crop_slices_from_masks(
            [gt_mask], margin=CROP_MARGIN_PX
        )
    else:
        ys, xs = slice(None), slice(None)

    rows.append({
        "department": item["department"],
        "image": image_np,
        "valid": valid_np,
        "valid_mask": valid_mask,
        "crop": (ys, xs),
        "rgb": rgb[ys, xs],
        "gt": gt_mask[ys, xs],
        "predictions": {},
    })

if not rows:
    raise ValueError("No se seleccionaron parches para la visualización.")

# Los modelos se cargan secuencialmente para limitar el uso de memoria.
for model_name in MODEL_NAMES:
    model2 = load_best_model(model_name).to(DEVICE).eval()

    for row in rows:
        image_t = torch.as_tensor(
            row["image"][None], device=DEVICE
        ).float()
        valid_t = torch.as_tensor(
            row["valid"][None], device=DEVICE
        ).float()

        with torch.inference_mode():
            logits, _ = model2(image_t, valid_t)
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

        pred = (prob > THRESHOLD) & row["valid_mask"]
        ys, xs = row["crop"]
        row["predictions"][model_name] = pred[ys, xs]

        del image_t, valid_t, logits

    model2.cpu()
    del model2
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# La figura comparte las columnas de imagen y referencia.
ncols = 2 + len(MODEL_NAMES)
fig, axes = plt.subplots(
    nrows=len(rows),
    ncols=ncols,
    figsize=(2.4 * ncols, 2.4 * len(rows)),
    dpi=300,
    squeeze=False,
)

titles = ["RGB-NIR", "Ground Truth"] + [
    MODEL_CONFIGS[name]["label"] for name in MODEL_NAMES
]

for col, title in enumerate(titles):
    axes[0, col].set_title(title, fontsize=8, pad=5)

for row_idx, row in enumerate(rows):
    axes[row_idx, 0].imshow(row["rgb"])
    axes[row_idx, 0].axis("off")

    _overlay_mask(
        axes[row_idx, 1],
        row["rgb"],
        row["gt"],
        (0.0, 0.85, 0.25, 0.55),
    )

    for col, model_name in enumerate(MODEL_NAMES, start=2):
        _overlay_mask(
            axes[row_idx, col],
            row["rgb"],
            row["predictions"][model_name],
            (1.0, 0.12, 0.04, 0.55),
        )

    axes[row_idx, 0].text(
        -0.08, 0.5,
        row["department"],
        transform=axes[row_idx, 0].transAxes,
        rotation=90,
        va="center",
        ha="right",
        fontsize=9,
        fontweight="bold",
    )

plt.subplots_adjust(
    left=0.055,
    right=0.995,
    top=0.94,
    bottom=0.02,
    wspace=0.035,
    hspace=0.09,
)

figure_dir = PERSISTENT_ROOT / "qualitative_figures"
figure_dir.mkdir(parents=True, exist_ok=True)

figure_name = "qualitative_comparison_all_models"
fig.savefig(figure_dir / f"{figure_name}.pdf", bbox_inches="tight")
fig.savefig(
    figure_dir / f"{figure_name}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)
print("Figuras:", figure_dir)